Imports

In [ ]:
import json
import gzip
import os
import re
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score, mean_squared_error
import pickle
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Data Loading

In [ ]:
def load_gz_json(filepath, max_samples=13000):
    # load reviews from gz file
    records = []
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if 'reviewText' in obj and 'overall' in obj:
                    records.append(obj)
            except:
                continue
            if len(records) >= max_samples:
                break
    return records

# file paths
beauty_path    = 'Dataset/Beauty_5.json.gz'
phones_path    = 'Dataset/Cell_Phones_and_Accessories_5.json.gz'
sports_path    = 'Dataset/Sports_and_Outdoors_5.json.gz'

print('Loading Beauty...')
beauty_data  = load_gz_json(beauty_path,  13000)
print(f'Beauty: {len(beauty_data)}')

print('Loading Cell Phones...')
phones_data  = load_gz_json(phones_path,  13000)
print(f'Phones: {len(phones_data)}')

print('Loading Sports...')
sports_data  = load_gz_json(sports_path,  13000)
print(f'Sports: {len(sports_data)}')

all_data = beauty_data + phones_data + sports_data
random.shuffle(all_data)
print(f'Total: {len(all_data)}')

Preprocessing

In [ ]:
def clean_text(text):
    # basic text cleaning
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    # whitespace tokenization
    return text.split()

def map_sentiment(rating):
    # map rating to 3 classes
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

def review_length_class(tokens):
    # derived feature: review length bucket
    # 0=short(<20), 1=medium(20-60), 2=long(>60)
    n = len(tokens)
    if n < 20:
        return 0
    elif n <= 60:
        return 1
    else:
        return 2

# clean and tokenize all samples
samples = []
for item in all_data:
    text   = clean_text(item['reviewText'])
    tokens = tokenize(text)
    if len(tokens) < 3:
        continue
    sentiment   = map_sentiment(int(item['overall']))
    len_class   = review_length_class(tokens)
    samples.append({'tokens': tokens, 'sentiment': sentiment, 'len_class': len_class})

random.shuffle(samples)
print(f'Cleaned samples: {len(samples)}')

Train Val Test Split

In [ ]:
# split 70 15 15
n = len(samples)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

train_samples = samples[:n_train]
val_samples   = samples[n_train:n_train + n_val]
test_samples  = samples[n_train + n_val:]

print(f'Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}')

Vocabulary

In [ ]:
# build vocab from training set only
MIN_FREQ  = 3
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

counter = Counter()
for s in train_samples:
    counter.update(s['tokens'])

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
print(f'Vocab size: {VOCAB_SIZE}')

def tokens_to_ids(tokens, vocab, max_len=128):
    # convert tokens to indices with padding and truncation
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens[:max_len]]
    if len(ids) < max_len:
        ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return ids

MAX_LEN = 128

Dataset

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, samples, vocab, max_len=128):
        self.data = samples
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        s = self.data[idx]
        input_ids   = tokens_to_ids(s['tokens'], self.vocab, self.max_len)
        # pad mask: 1 for real tokens, 0 for padding
        attn_mask   = [1 if x != 0 else 0 for x in input_ids]
        return (
            torch.tensor(input_ids,   dtype=torch.long),
            torch.tensor(attn_mask,   dtype=torch.float),
            torch.tensor(s['sentiment'], dtype=torch.long),
            torch.tensor(s['len_class'], dtype=torch.long)
        )

BATCH_SIZE = 64

train_ds = ReviewDataset(train_samples, vocab, MAX_LEN)
val_ds   = ReviewDataset(val_samples,   vocab, MAX_LEN)
test_ds  = ReviewDataset(test_samples,  vocab, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('Dataloaders ready')

Transformer Architecture

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # compute positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # add positional encoding
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k, dropout=0.1):
        super().__init__()
        self.d_k    = d_k
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        # scaled dot product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        return torch.matmul(attn, V), attn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads
        # separate linear projections for each head
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention(self.d_k, dropout)
        self.dropout   = nn.Dropout(dropout)

    def split_heads(self, x):
        # reshape to (batch, heads, seq, d_k)
        B, T, D = x.size()
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        B, T, _ = x.size()
        Q = self.split_heads(self.W_Q(x))
        K = self.split_heads(self.W_K(x))
        V = self.split_heads(self.W_V(x))
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)  # broadcast over heads
        out, _ = self.attention(Q, K, V, mask)
        # combine heads
        out = out.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.d_k)
        return self.W_O(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff      = FeedForward(d_model, d_ff, dropout)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # self attention with residual
        x = self.norm1(x + self.dropout(self.attn(x, mask)))
        # feedforward with residual
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers,
                 d_ff, max_len, num_sentiment_classes, num_len_classes, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model, max_len, dropout)
        self.layers    = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        # task heads
        self.sentiment_head = nn.Linear(d_model, num_sentiment_classes)
        self.len_head       = nn.Linear(d_model, num_len_classes)

    def forward(self, input_ids, attn_mask=None):
        # embed and encode
        x = self.embedding(input_ids)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, attn_mask)
        x = self.norm(x)
        # mean pooling over non-padding tokens
        if attn_mask is not None:
            mask_expanded = attn_mask.unsqueeze(-1).float()
            pooled = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        else:
            pooled = x.mean(dim=1)
        # classification heads
        sentiment_logits = self.sentiment_head(pooled)
        len_logits       = self.len_head(pooled)
        return sentiment_logits, len_logits, pooled

print('Architecture defined')

Model Init

In [ ]:
# model hyperparameters
D_MODEL    = 128
NUM_HEADS  = 4
NUM_LAYERS = 2
D_FF       = 256
DROPOUT    = 0.1

model = TransformerEncoder(
    vocab_size            = VOCAB_SIZE,
    d_model               = D_MODEL,
    num_heads             = NUM_HEADS,
    num_layers            = NUM_LAYERS,
    d_ff                  = D_FF,
    max_len               = MAX_LEN,
    num_sentiment_classes = 3,
    num_len_classes       = 3,
    dropout               = DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {total_params:,}')